In [18]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd

In [4]:
movies = pd.read_csv("/Users/sophie/Dev/movie-recommender-system/data/movies_cleaned_up.csv")

In [5]:
movies

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
...,...,...,...,...,...,...,...,...,...
9980,10196,The Last Airbender,"Action,Adventure,Fantasy",en,"The story follows the adventures of Aang, a yo...",98.322,2010-06-30,4.7,3347
9981,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventure",en,The sharks take bite out of the East Coast whe...,12.490,2015-07-22,4.7,417
9982,13995,Captain America,"Action,Science Fiction,War",en,"During World War II, a brave, patriotic Americ...",18.333,1990-12-14,4.6,332
9983,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama",en,A man named Farmer sets out to rescue his kidn...,15.159,2007-11-29,4.7,668


In [63]:
# create combined text for title, genre, and overview fields
movies["combined_text"] = (
    #"Title: " + movies["title"].fillna("") + ". " +
    "Genre: " + movies["genre"].fillna("") + ". " +
    "Overview: " + movies["overview"].fillna("")
)
movies["combined_text"]

0       Genre: Drama,Crime. Overview: Framed in the 19...
1       Genre: Comedy,Drama,Romance. Overview: Raj is ...
2       Genre: Drama,Crime. Overview: Spanning the yea...
3       Genre: Drama,History,War. Overview: The true s...
4       Genre: Drama,Crime. Overview: In the continuin...
                              ...                        
9980    Genre: Action,Adventure,Fantasy. Overview: The...
9981    Genre: Action,TV Movie,Science Fiction,Comedy,...
9982    Genre: Action,Science Fiction,War. Overview: D...
9983    Genre: Adventure,Fantasy,Action,Drama. Overvie...
9984    Genre: Thriller,Action,Crime. Overview: Seekin...
Name: combined_text, Length: 9985, dtype: str

In [64]:
# get embeddings - using hf transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    movies["combined_text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [65]:
embeddings

array([[-0.01507484, -0.05933658, -0.06913045, ...,  0.04426529,
         0.08803046, -0.05934433],
       [-0.0237387 ,  0.02333198, -0.04900789, ...,  0.02851259,
         0.00259286, -0.01558666],
       [-0.07220379,  0.03323361, -0.06877995, ...,  0.01351652,
         0.08967282, -0.05792296],
       ...,
       [-0.05933784, -0.0218488 , -0.04173438, ..., -0.02284305,
        -0.04142766, -0.00319057],
       [-0.08427592, -0.03778565, -0.02804293, ..., -0.02913866,
         0.01068622,  0.01664642],
       [-0.00747918,  0.01418289, -0.06008193, ...,  0.01504259,
         0.05038825, -0.00843891]], shape=(9985, 384), dtype=float32)

In [66]:
print(embeddings.shape)
print(movies.shape)

(9985, 384)
(9985, 10)


In [76]:
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

def recommend_movies(movie_title, movies, embeddings, top_n=10):
    matches = movies[movies["title"].str.lower() == movie_title.lower()] # find the movie row(s) that matches the user input
   
    if matches.empty:
        return None
    
    idx = matches.index[0] # gets the row of the movie in movies df
    query_vector = embeddings[idx].reshape(1, -1) # gets the movie embedding vector and changes shape for cosine similairty to work with

    similarities = cosine_similarity(query_vector, embeddings)[0] # compares with all the movie's embeddings and gets the first row from result

    top_indices = np.argsort(similarities)[::-1] # sorts movies by similarity scores
    top_indices = [i for i in top_indices if i != idx][:top_n] # returns top n movies (besides itself)

    return movies.iloc[top_indices][["title", "genre", "overview"]].assign(
        similarity=similarities[top_indices]
    )

In [77]:
recommend_movies("Girl, Interrupted", movies, embeddings)

,title,genre,overview,similarity
2186,Nomadland,Drama,A woman in her sixties embarks on a journey th...,0.614804
441,About Time,"Drama,Romance,Fantasy",The night after another unsatisfactory New Yea...,0.584199
1447,The First Day of the Rest of Your Life,Drama,A sprawling drama centered on five key days in...,0.581194
1964,The Hours,Drama,"""The Hours"" is the story of three women search...",0.577275
884,"Synecdoche, New York",Drama,"A theater director struggles with his work, an...",0.576784
1553,Alice,"Animation,Fantasy,Adventure",A quiet young English girl named Alice finds h...,0.575893
450,Letter from an Unknown Woman,"Drama,Romance",A pianist about to flee from a duel receives a...,0.575026
6074,My Blueberry Nights,"Drama,Romance",Elizabeth has just been through a particularly...,0.566208
9075,Horse Girl,Drama,A socially awkward woman with a fondness for a...,0.563820
1299,Happiness,"Comedy,Drama",The lives of many individuals connected by the...,0.562612


In [78]:
recommend_movies("Harry Potter and the Deathly Hallows: Part 2", movies, embeddings)

,title,genre,overview,similarity
664,Harry Potter and the Deathly Hallows: Part 1,"Adventure,Fantasy","Harry, Ron and Hermione walk away from their l...",0.740734
765,Harry Potter and the Chamber of Secrets,"Adventure,Fantasy","Cars fly, trees fight back, and a mysterious h...",0.594391
432,Harry Potter and the Philosopher's Stone,"Adventure,Fantasy",Harry Potter has lived under the stairs at his...,0.580858
805,Harry Potter and the Half-Blood Prince,"Adventure,Fantasy",As Lord Voldemort tightens his grip on both th...,0.580234
829,Harry Potter and the Order of the Phoenix,"Adventure,Fantasy,Mystery",A summer has passed since Harry's encounter wi...,0.568440
3480,The Wizards Return: Alex vs. Alex,"Family,Adventure,Comedy,Drama","While visiting Italy with her family, a young ...",0.555443
7523,Ronal the Barbarian,"Animation,Adventure,Fantasy",Ronal is a young barbarian with low self-estee...,0.542158
584,Harry Potter and the Goblet of Fire,"Adventure,Fantasy,Family",When Harry Potter's name emerges from the Gobl...,0.529110
2558,My Little Pony: The Movie,"Family,Animation,Music,Adventure,Fantasy","A new dark force threatens Ponyville, and the ...",0.524083
302,Harry Potter and the Prisoner of Azkaban,"Adventure,Fantasy",Year three at Hogwarts means new fun and chall...,0.515554


In [70]:
recommend_movies("Point Break", movies, embeddings)

,title,genre,overview,similarity
8382,Marauders,"Action,Crime,Thriller",An untraceable group of elite bank robbers is ...,0.731521
4379,Den of Thieves,"Action,Thriller,Crime",A gritty crime saga which follows the lives of...,0.723395
1653,Inside Man,"Crime,Drama,Thriller","When an armed, masked gang enter a Manhattan b...",0.693940
6865,The Crew,"Drama,Crime,Thriller",One of the members of a gang of thieves commit...,0.686788
5469,2 Guns,"Action,Comedy,Crime",A DEA agent and an undercover Naval Intelligen...,0.661556
2531,Set It Off,"Crime,Drama,Thriller","Four black women, all of whom have suffered fo...",0.658055
1216,Donnie Brasco,"Crime,Drama,Thriller",An FBI undercover agent infilitrates the mob a...,0.647514
9792,The Trust,"Crime,Thriller",A pair of cops investigating a drug invasion s...,0.647231
661,L.A. Confidential,"Crime,Drama,Mystery,Thriller",Three detectives in the corrupt and brutal L.A...,0.645103
9148,Suspect Zero,"Crime,Thriller","A killer is on the loose, and an FBI agent sif...",0.644738


In [61]:
import numpy as np

np.save("movie_embeddings.npy", embeddings)
movies.to_csv("movies_processed.csv", index = False)